In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch

In [ ]:
all_sheets = pd.read_excel("/content/LK_modified_1.xlsx", sheet_name=None)

# Показать список листов
print(all_sheets.keys())  # Например: ['Лист1', 'Лист2']

# Доступ к конкретному листу
df_phr = all_sheets['Популярные фразы']
df_qa = all_sheets['Вопрос ответ']
df_glos = all_sheets['Глоссарий']

dict_keys(['Популярные фразы', 'Вопрос ответ', 'Глоссарий'])


In [ ]:
df_phr.head()

,Ответ,Вопрос,Unnamed: 2
0,Для оформления обращения в техническую поддерж...,выйти,другое
1,Чат-бот находится в стадии пилотирования и обу...,Уберите этот чат бот он очень мешает,жалоба
2,Чат-бот находится в стадии пилотирования и обу...,вернуть старый дизайн,жалоба
3,Для оформления обращения в техническую поддерж...,хочу создать заявку,заявка
4,Для оформления обращения в техническую поддерж...,мне нужно создать заявку,заявка


In [ ]:
df_qa.head()

,Unnamed: 0,question,content,category
0,0,"Я сменил автомобить, на учет еще не поставил, ...",Для внесения данных по личному автомобилю обра...,автомобиль
1,1,Не отображается автомобиль в личном кабинете.,Для внесения данных по личному автомобилю обра...,автомобиль
2,2,добавить автомобиль,Для внесения данных по личному автомобилю обра...,автомобиль
3,3,хочу внести данные об автомобиле,Для внесения данных по личному автомобилю обра...,автомобиль
4,4,Как внести данные об автомобиле?,Для внесения данных по личному автомобилю обра...,автомобиль


In [ ]:
df_glos.head()

,Сокращение,Расшифровка
0,лк,личный кабинет
1,БиР,Беременность и роды
2,зп,заработная плата
3,НДФЛ,Налог на доходы физических лиц
4,СТД,срочный трудовой договор


In [ ]:
df_qa["category"] = df_qa["category"].apply(lambda x: x.lower())
df_qa["question"] = df_qa["question"].apply(lambda x: x.lower())

In [ ]:
dic = dict()

In [ ]:
for cat in df_qa["category"].unique():
    dic[cat] = len(df_qa[df_qa["category"] == cat]["content"].unique()), len(df_qa[df_qa["category"] == cat]["content"])

In [ ]:
sorted_dict = sorted(dic.items(), key=lambda x: x[1])
for i in sorted_dict:
    print(i[0], i[1])

мчд (1, 1)
сб (1, 1)
sed (1, 1)
выручай-карта (1, 1)
перевод (1, 2)
командировка (1, 5)
обучение (1, 7)
автомобиль (1, 8)
уход за больным (1, 9)
оператор (2, 5)
дмс (3, 3)
справка (3, 9)
налоговый вычет (3, 12)
моя карьера (3, 56)
табель (4, 168)
доверенность (5, 7)
график работы (5, 28)
материальная помощь (6, 8)
отгул (7, 48)
заявки (8, 47)
поддержка (8, 276)
больничный (9, 17)
документооборот (9, 22)
удаленная работа (11, 98)
эцп (12, 33)
зарплата (13, 45)
увольнение (16, 60)
прием на работу (19, 48)
бир (22, 50)
лк (27, 481)
отпуск (39, 120)


In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("Tochka-AI/ruRoPEBert-e5-base-2k", trust_remote_code=True, output_hidden_states=True)
model = AutoModelForMaskedLM.from_pretrained("Tochka-AI/ruRoPEBert-e5-base-2k", trust_remote_code=True, output_hidden_states=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

...

modeling_rope_bert.py:   0%|          | 0.00/46.2k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Tochka-AI/ruRoPEBert-e5-base-2k:
- modeling_rope_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/556M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:820: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_hidden_states` is. When `return_dict_in_generate` is not `True`, `output_hidden_states` is ignored.
  warnings.warn(
Some weights of RoPEBertForMaskedLM were not initialized from the model checkpoint at Tochka-AI/ruRoPEBert-e5-base-2k and are newly initialized: ['cls.predictions.decoder.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Загрузка модели и токенизатора
tokenizer = AutoTokenizer.from_pretrained("Tochka-AI/ruRoPEBert-e5-base-2k", trust_remote_code=True)
model = AutoModel.from_pretrained("Tochka-AI/ruRoPEBert-e5-base-2k", trust_remote_code=True)

def get_embeddings(texts):
    """Преобразует список текстов в эмбеддинги"""
    embeddings = []
    for i in range(0, len(texts)):
        print(i)
        batch = texts[i]


        # Токенизация и получение эмбеддингов
        inputs = tokenizer(batch, padding=True, truncation=True,
                          max_length=512, return_tensors="pt")

        with torch.no_grad():
            outputs = model(**inputs)

        # Используем среднее значение скрытых состояний как эмбеддинг
        batch_embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        embeddings.extend(batch_embeddings)

    return np.array(embeddings)

# Пример текстов для сравнения
texts = list(df_qa["question"])

# Получаем эмбеддинги
embeddings = get_embeddings(texts)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [ ]:
# Сравниваем первый текст со всеми остальными
query = "пособие"
query_embedding = get_embeddings([query])

# Вычисляем косинусную близость
similarities = cosine_similarity(query_embedding, embeddings)[0]

# Сортируем результаты
results = sorted(zip(texts, similarities), key=lambda x: x[1], reverse=True)

# Выводим результаты
print(f"Запрос: '{query}'\n")
print("Самые похожие тексты:")
for text, score in results[:6]:
    print(f"{score:.4f}: {text}")

0
Запрос: 'пособие'

Самые похожие тексты:
0.6709: изменить данные пособия
0.6593: премия
0.6438: курсы
0.6415: обучение
0.6080: надбавка
0.6080: надбавка


In [ ]:
similarities

array([0.21683227, 0.2506351 , 0.27226418, ..., 0.24062645, 0.26617807,
       0.31197298], dtype=float32)

In [ ]:
texts

['я сменил автомобить, на учет еще не поставил, могу ли я заправляться по топливной карте?',
 'не отображается автомобиль в личном кабинете.',
 'добавить автомобиль',
 'хочу внести данные об автомобиле',
 'как внести данные об автомобиле?',
 'мне нужно внести данные об автомобиле',
 'внести машину',
 'при смене автомобиля, который используется в служебных целях, как изменить марку и госномер авто в личном кабинете?',
 'будет ли оплачен больничный, если он открыт в другой стране?',
 'как оформить больничный по беременности и родам',
 'что делать после получения больничного по беременности и родам',
 'как оформить бир?',
 'как оформить больничный по беременности и родам?',
 'как оформить моей сотруднице декретный отпуск?',
 'сотрудница уходит в декрет, как мне провести ей декретный отпуск',
 'как оформить пособие по рождению ребенка?',
 'родила ребенка, как оформить пособие?',
 'хочу оформить единовременную выплату',
 'какие документы нужно подать на отпуск по уходу за ребенком?',
 'каки